# Exercise: Regularise the Forest Cover Classifier

The Covertype data describes forest areas by their terrain, and the task is to predict the tree species that grows there. A baseline network with two hidden layers reaches a validation accuracy of about 0.82 after 30 epochs, and its learning curves are still improving. In this exercise you use what you learnt today to improve it: you train it longer, make it larger, diagnose its learning curves and regularise it, comparing every version on the validation set. At the end you evaluate your chosen model once on the test set.

The tasks describe what to build and what to report; the code is yours to write. Notebooks 01 and 02 contain the training settings and the regularisation tools you need; the data preparation and the accuracy metric are those you used for multiclass classification before. A larger network trains for one to two minutes, so plan your runs before you start them.

## Learning Objectives

By the end of this exercise, you should be able to:

- Train a Keras classifier on real data and diagnose its learning curves for loss and accuracy.
- Apply L2 regularisation, dropout and early stopping to a larger network and compare the versions on a validation set.
- Choose the value to monitor in `EarlyStopping` and `ModelCheckpoint` that matches the goal of the model.
- Evaluate the chosen model once on the test set and compare it with the baseline network.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: Load the Data

The data is a 60,000-row sample of the [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype), loaded with [`fetch_covtype`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html): 54 features (10 terrain measurements and 44 one-hot encoded wilderness and soil columns) and seven tree species as the target. The cell below loads it.

In [ ]:
covtype = fetch_covtype(as_frame=True)
forest = covtype.frame.sample(60_000, random_state=42)

class_names = [
    "Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow",
    "Aspen", "Douglas-fir", "Krummholz",
]
X = forest.drop(columns="Cover_Type")
y = forest["Cover_Type"] - 1  # codes 1-7 become labels 0-6
print("Rows and columns:", X.shape)

## Step 2: Split the Data and Prepare the Normalisation

Split the data as follows, so your numbers are comparable with the solution:

- First keep 20 % of the rows as the test set, then 20 % of the rest as the validation set.
- Use `stratify` and `random_state=42` in both splits.
- Print the shapes of `X_train`, `X_valid` and `X_test`.

Then create a [`Normalization`](https://keras.io/api/layers/preprocessing_layers/numerical/normalization/) layer and adapt it to the training features. Every model in this exercise reuses this layer.

## Step 3: Plan How You Train

You will train about ten versions of the same classifier. How you organise the code is up to you:

- **Write helper functions** (optional), similar to those in notebook 02. For example:
  1. `build_model(units, l2_strength=0.0, dropout_rate=0.0)` returns a new `Sequential` model: the input for 54 features, the normalisation layer, one ReLU `Dense` layer per entry of `units` (with L2 regularisation and a `Dropout` layer after each when the arguments are above 0), and an output layer for the seven classes.
  2. `train_model(model, epochs, callbacks=None)` compiles and trains the model and returns the history as a DataFrame.
  3. `plot_curves(logs, title)` draws the training and validation loss next to the training and validation accuracy.
- **Write each model out in full**, and copy and change the cells from one step to the next.

Whichever way you choose, keep these settings the same for every version, so the comparison is fair and your numbers are close to the solution's:

- Call `keras.utils.clear_session()` and `keras.utils.set_random_seed(42)` before you build each model.
- Choose the output layer and the loss for a multiclass problem with integer labels (notebook 01, section 2), and use them for every version.
- Compile with Adam (learning rate 0.001) and the accuracy metric.
- Train with `batch_size=128` and the validation set as `validation_data`.
- Plot the loss and the accuracy curves of every run, for both the training and the validation set.

Also create an empty dictionary `results`. After every version, store its validation accuracy there under a descriptive name, so you can compare all versions in step 9.

The cell below is for your helper functions, if you write them; you can also leave it empty.

## Step 4: The Baseline Network, Trained Longer

Build the baseline network: two hidden layers of 128 and 64 neurons, no regularisation.

1. Train it for 30 epochs and store its validation accuracy. It should be close to 0.82.
2. Build it again and train it for 100 epochs. Plot the learning curves and store the validation accuracy.

Which panel of the learning-curves figure in notebook 02 describes the 100-epoch run best? How much did the extra 70 epochs improve the validation accuracy?

*Your answers:*

## Step 5: A Larger Network

The validation accuracy is still rising slowly, so give the network more capacity: three hidden layers of 512, 256 and 128 neurons. Train it for 100 epochs, plot the learning curves and store the validation accuracy.

Look at both plots:

- At which epoch is the validation loss lowest, and what does it do afterwards?
- What does the validation accuracy do over the same epochs?
- Which panel of the learning-curves figure describes the loss curves? How can the validation loss rise while the validation accuracy also rises? Hint: accuracy only counts whether the class with the highest probability is correct, while cross-entropy also measures how much probability the model gives to the correct class.

*Your answers:*

## Step 6: L2 Regularisation

Train the large network for 100 epochs with L2 regularisation in every hidden layer. Try two strengths, $\lambda = 0.0001$ and $\lambda = 0.001$. Plot the learning curves of each run and store the validation accuracies.

Compare the curves with those of step 5: does L2 reduce the gap between training and validation accuracy? Does it improve the validation accuracy? Keras adds the L2 penalty to the reported loss, so compare the accuracy curves, not the loss values, with step 5.

*Your answers:*

## Step 7: Dropout

Train the large network for 100 epochs with dropout after every hidden layer. Try two rates, 0.2 and 0.4. Plot the learning curves of each run and store the validation accuracies.

Which rate works better? Are the curves still improving at epoch 100?

*Your answers:*

## Step 8: Early Stopping: What Should It Watch?

Train the large network without regularisation twice with an [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) callback (at most 100 epochs, `patience=10`, `restore_best_weights=True`):

1. once with `monitor="val_loss"`,
2. once with `monitor="val_accuracy"`.

For each run, print the number of epochs it trained and store the validation accuracy of the restored model.

Which of the two monitors fits the goal of this model, which is to predict the correct species as often as possible? Use your answer to step 5.

*Your answers:*

## Step 9: Your Final Model

Combine what worked best in steps 6 to 8 into one final model:

- Choose the regularisation from your results.
- Train for up to 200 epochs with early stopping (`patience=20`), so the model can train for longer while it improves.
- Save the best epoch with a [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/) to `models/forest_cover_best.keras`, monitoring the same value as the early stopping.
- Load the saved model with `keras.saving.load_model()`, store its validation accuracy and plot the learning curves.

Then show all your versions in one table, sorted by validation accuracy.

## Step 10: Evaluate Once on the Test Set

Evaluate your final model on the test set. The choice is already made, so also report the test accuracy of the 30-epoch baseline network from step 4 for reference only. For comparison, logistic regression reaches a test accuracy of 0.724 on this split.

Then calculate the accuracy of the final model within each of the seven species, the share of test rows of that species it predicts correctly (the **recall** of the class). For the baseline network, the three rare species are the weakest: Cottonwood/Willow 0.38, Aspen 0.47 and Douglas-fir 0.51. Have regularisation and the larger network helped them?

*Your answers:*

## Reflection

Answer these questions in the Markdown cell below.

1. In step 5 the validation loss and the validation accuracy pointed in different directions. Which one would you use to choose a model for this problem, and why?
2. Which regularisation technique helped most here, and what did its learning curves look like compared with the large network without regularisation?
3. Every decision in this exercise was made on the validation set. Why does that make the validation accuracy of your final model slightly optimistic, and how does the test set solve this?
4. Your final model is larger and trains longer than the baseline network. Name one situation in which you would still choose the smaller network.

*Your answers:*

## References

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 11: Avoiding Overfitting Through Regularization.
- Keras API: [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/), [layer weight regularisers](https://keras.io/api/layers/regularizers/), [`EarlyStopping`](https://keras.io/api/callbacks/early_stopping/) and [`ModelCheckpoint`](https://keras.io/api/callbacks/model_checkpoint/).
- Guo, C., Pleiss, G., Sun, Y. and Weinberger, K. Q. (2017). [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html). *Proceedings of the 34th International Conference on Machine Learning*: why large networks become overconfident while their accuracy still improves.
- Blackard, J. (1998). [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype). UCI Machine Learning Repository.